# Week 23 · Notebook 2: AI Search (Vector Search) RAG over Policy Docs

# Requirements: Databricks workspace (free trial)

No CSV upload is required, the policy documents are embedded inline below as a Delta table. AI Search (Vector Search) runs on **serverless** infrastructure, and `ai_query` requires **serverless compute + DBR 18.2+**.


## AI Search + RAG

**AI Search** (formerly Vector Search) manages vector retrieval in Unity Catalog. We create an **index from a Delta table** (Delta Sync), then run **similarity** (ANN) and **hybrid** (ANN + BM25 fused by Reciprocal Rank Fusion) search, and finally generate a grounded answer with `ai_query`. See research §7.3 and `reference/platforms/databricks/11-vector-search-rag.md`.


In [ ]:
from pyspark.sql import Row

# The policy corpus (from zoro/data.py policy_docs, inlined for the workspace).
docs = [
  ("POL-001", "Shipping & Delivery Policy",
   "ZoroLogistics Shipping & Delivery Policy. Standard ground transit is 2-7 business "
   "days depending on lane distance. Every shipment receives a tracking id at booking. "
   "During declared severe weather, delivery SLAs are extended by 48 hours without "
   "penalty. Address changes are free before pickup and $85 after pickup."),
  ("POL-002", "Refund & Claims Policy",
   "ZoroLogistics Refund & Claims Policy. Shipments arriving more than 48 hours late are "
   "eligible for a 10% freight refund; more than 7 days late, a 50% freight refund. Damage "
   "claims must be filed within 7 days of delivery with photo evidence; approved claims "
   "refund the declared value up to $5,000. Refunds over $500 require supervisor approval."),
  ("POL-003", "Dangerous Goods Policy",
   "ZoroLogistics Dangerous Goods Policy. Lithium batteries over 100 Wh, explosives, and "
   "unapproved chemicals cannot be shipped. DG shipments require a signed shipper's "
   "declaration and UN number on the bill of lading. Vehicles carrying DG must display "
   "placards and follow hazmat routing."),
  ("POL-004", "Customs & Border Policy",
   "ZoroLogistics Customs & Border Policy. Cross-border shipments require a commercial "
   "invoice and bill of lading; missing documents add 1-3 days at the border. Duties and "
   "taxes are the consignee's responsibility unless prepaid at booking. Customs holds "
   "beyond 5 days incur a $40/day storage fee."),
]

rows = [Row(doc_id=d, title=t, text=x) for d, t, x in docs]
spark.createDataFrame(rows).write.mode("overwrite").saveAsTable("zrl_.zorologistics.policy_docs")
print("policy docs:", len(docs))


## Embed the docs with a foundation-model embedding function

We call a **Foundation Model API** embedding endpoint through `mlflow.deployments`. The endpoint name (`databricks-gte-large-en`) is a Databricks-hosted foundation model, pay-per-token, so mind the size of your corpus. The returned vectors are stored in an `embedding` array column (self-managed embeddings).


In [ ]:
import mlflow.deployments

deploy_client = mlflow.deployments.get_deploy_client("databricks")
EMBED_ENDPOINT = "databricks-gte-large-en"   # foundation-model embedding endpoint

def embed(texts):
    # Foundation Model API (OpenAI-compatible). Returns {"data": [{"embedding": [...]}, ...]}.
    resp = deploy_client.predict(endpoint=EMBED_ENDPOINT, inputs={"input": texts})
    return [d["embedding"] for d in resp["data"]]

pdf = spark.table("zrl_.zorologistics.policy_docs").toPandas()
pdf["embedding"] = embed(pdf["text"].tolist())
spark.createDataFrame(pdf).write.mode("overwrite").saveAsTable("zrl_.zorologistics.policy_docs")
print("embedded rows:", len(pdf), "| vector dim:", len(pdf["embedding"].iloc[0]))


## Create the AI Search index (Delta Sync) via the SDK

`VectorSearchClient.create_delta_sync_index` registers a Delta Sync index that auto-syncs from the source table. We use the self-managed `embedding_vector_column` (we computed embeddings above); the alternative is *managed* embeddings (provide `embedding_source_column` + `embedding_model_endpoint_name` and let the index embed).


In [ ]:
from databricks.vector_search.client import VectorSearchClient

vsc = VectorSearchClient()
ENDPOINT = "zrl_policy_endpoint"
SOURCE = "zrl_.zorologistics.policy_docs"

# The endpoint must exist before the index.
try:
    vsc.create_endpoint(name=ENDPOINT, endpoint_type="STANDARD")
except Exception as e:
    print("endpoint note:", str(e)[:100])

index = vsc.create_delta_sync_index(
    endpoint_name=ENDPOINT,
    source_table_name=SOURCE,
    pipeline_type="TRIGGERED",          # incremental sync on source change
    primary_key="doc_id",
    embedding_vector_column="embedding",  # self-managed embeddings
)
print("index status:", index.describe().get("status"))


## Similarity search (ANN)

Embed the query with the *same* function, then run a pure vector nearest-neighbor search.


In [ ]:
query = "How do I file a damage claim for a late shipment?"
q_emb = embed([query])[0]

res = index.similarity_search(
    query_vector=q_emb,
    columns=["doc_id", "title", "text"],
    num_results=3,
)

print("Query:", query)
for row in res.get("result", {}).get("data_array", []):
    print("  -", row[0], "|", row[1])


## Hybrid search (ANN + BM25)

Hybrid search fuses vector similarity with keyword (BM25) matching via **Reciprocal Rank Fusion**. Passing both `query_text` (for BM25) and `query_vector` (for the vector side) makes the fusion explicit.


In [ ]:
res_hybrid = index.similarity_search(
    query_text=query,
    query_vector=q_emb,
    query_type="HYBRID",
    columns=["doc_id", "title", "text"],
    num_results=3,
)

print("Hybrid results for:", query)
retrieved = res_hybrid.get("result", {}).get("data_array", [])
for row in retrieved:
    print("  -", row[0], "|", row[1])


## Build the RAG answer with `ai_query`

Assemble the retrieved docs into a context block, then call `ai_query` (serverless + DBR 18.2+) against a foundation-model chat endpoint. The prompt instructs the model to answer *only* from the context and cite doc ids.


In [ ]:
def aiq(prompt, endpoint="databricks-meta-llama-3-3-70b-instruct"):
    p = prompt.replace("'", "''")
    return spark.sql(f"SELECT ai_query('{endpoint}', '{p}') AS answer").collect()[0][0]

context = "\n\n".join([f"[{r[0]}] {r[2]}" for r in retrieved])
prompt = (
    "You are a ZoroLogistics support assistant. Answer ONLY from the context below, "
    "and cite the document ids you used.\n\nContext:\n" + context +
    "\n\nQuestion: " + query
)

answer = aiq(prompt)
print(answer)


In [ ]:
# Groundedness: fraction of the answer's tokens that appear in the retrieved context.
import re

def groundedness(answer, context):
    a_tokens = set(re.findall(r"[a-z0-9]+", answer.lower()))
    c_tokens = set(re.findall(r"[a-z0-9]+", context.lower()))
    if not a_tokens:
        return 0.0
    return round(len(a_tokens & c_tokens) / len(a_tokens), 4)

score = groundedness(answer, context)
print("retrieved docs:", len(retrieved))
print("groundedness score (token overlap, proxy):", score)
